# Haas Japanese Colab Notebook

Wannan notebook yana amfani da Third Intelligence **Haas-ja-v1.0** don rubuta fayilolin sauti da bidiyon Jafananci zuwa rubutu cikin sauri.

## Fasali

- **Kyauta:** Yana aiki a kan runtime na GPU kyauta na Google Colab.
- **Samfurin gane maganar Jafananci:** Yana amfani da Third Intelligence `third-intelligence/Haas-ja-v1.0`, samfurin kusan 0.6B parameters don mayar da maganar Jafananci zuwa rubutu.
- **Alamomin rubutu:** Samfurin yana fitar da alamomin rubutun Jafananci da aka kiyasta kamar `、` da `。`.
- **Lokuta da subtitles:** `subtitle` creates SRT with approximate chunk boundaries, not native word/character timestamps; `none` disables timing.
- **Dogayen fayiloli:** Ana raba dogayen fayilolin audio/video a ciki zuwa ƙananan audio chunks sannan a transcribe su bi da bi.
- **Google Drive:** Loda fayiloli kai tsaye ko sarrafa fayilolin da aka adana a Google Drive.
- **Tsarin fitarwa iri-iri:** Ana samar da TXT / Markdown / JSON / CSV / XLSX / SRT kuma a tattara su cikin archive ZIP.

## Amfani

1. Zaɓi `INPUT_MODE` a **1. Settings** sannan ka canza sauran saituna idan ya cancanta.
2. Zaɓi **Runtime → Run all**.
3. Idan `INPUT_MODE = "upload"` ne, zaɓi fayiloli a **2. File selection / mount na Google Drive**.
4. Ana sauke ZIP na fitarwa kai tsaye bayan sarrafawa ya ƙare.

## Bayanan kula

- Notebook yana tsayawa kafin transkripsan idan babu GPU.
- Lokacin sarrafawa yana canzawa. A farkon run, dole a sauke NeMo da samfurin kusan 1.23 GB a Colab kafin transkripsan ya fara. Direct uploads kuma suna buƙatar lokacin upload.
- Ana sarrafa dogon audio a cikin chunks na ciki, don haka context na iya katsewa a kusa da chunk boundaries kuma gane magana na iya shafuwa. Ƙara `CHUNK_SECONDS` yana rage yawan boundaries amma yana sa kowace inference ta fi nauyi; rage shi yana sa inference ta fi sauƙi amma yana ƙirƙirar ƙarin boundaries.
- Samfurin Third Intelligence yana ƙarƙashin lasisin **Apache-2.0**.


In [ ]:
# @title 1. Saituna
# ============================================================
# 1. Saituna
# ============================================================
import os
os.environ["USE_TF"] = "0"  # Set before importing Transformers.
from pathlib import Path

# @markdown ## Fayilolin sauti / bidiyo
# @markdown - `INPUT_MODE`: Yi amfani da `upload` don loda fayiloli kai tsaye, ko `google_drive` don karanta fayilolin da aka riga aka adana a Google Drive.
INPUT_MODE = "upload"  # @param ["upload", "google_drive"]

# @markdown ### Karanta fayiloli daga Google Drive
# @markdown Ana amfani da waɗannan saituna ne kawai idan `INPUT_MODE == "google_drive"`.
# @markdown - `GOOGLE_DRIVE_ROOT`: Yawanci ka bar wannan a matsayin `/content/drive/MyDrive`.
# @markdown - `GOOGLE_DRIVE_SUBPATH`: Path zuwa file ko folder a ƙarƙashin MyDrive. Ka bar shi babu komai don amfani da babban folder na MyDrive.
# @markdown - `RECURSIVE`: Idan Drive input path folder ne, bincika subfolders ma.
# @markdown - `FILENAME_CONTAINS`: Tacewar zaɓi don wani ɓangare na sunan fayil.
GOOGLE_DRIVE_ROOT = "/content/drive/MyDrive"  # @param {type:"string"}
GOOGLE_DRIVE_SUBPATH = ""  # @param {type:"string"}
RECURSIVE = False  # @param {type:"boolean"}
FILENAME_CONTAINS = ""  # @param {type:"string"}

# @markdown ## Decoder
# @markdown - Haas-ja-v1.0 uses TDT only; CTC is unavailable.
MODEL_ID = "third-intelligence/Haas-ja-v1.0"

# @markdown ## Lokuta
# @markdown - `TIMESTAMP_MODE`: `none` adds no timing information. `subtitle` creates SRT using approximate chunk boundaries, not native word timestamps.
TIMESTAMP_MODE = "subtitle"  # @param ["subtitle", "none"]

# @markdown ## Dogon sauti
# @markdown - `CHUNK_SECONDS`: Yana sarrafa seconds na dogon file da ake process a kowane internal chunk. Manyan values suna rage chunk boundaries kuma suna kiyaye ƙarin context, amma kowace inference ta fi nauyi kuma tana iya amfani da ƙarin GPU memory. Ƙananan values suna sa kowace inference ta fi sauƙi amma suna ƙirƙirar ƙarin boundaries, wanda zai iya katse recognition kusa da waɗannan points. Yawanci ka bar default value ba tare da canji ba. Ba a canza original input file ɗin kansa.
CHUNK_SECONDS = 28.0  # @param {type:"number"}
INFERENCE_BATCH_SIZE = 1

# Nunin subtitle (saitunan ciki)
SUBTITLE_MAX_LINE_CHARS = 20

# @markdown ## Fitarwa
# @markdown - `SAVE_OUTPUTS_TO_GOOGLE_DRIVE`: Idan `True`, ajiye output ZIP zuwa Google Drive ba tare da la’akari da input method ba.
# @markdown - `AUTO_DOWNLOAD_ZIP`: Ƙirƙiri archive ZIP guda ɗaya a ƙarshe sannan ka sauke shi.
# - Ana sarrafa output formats da waɗannan Boolean flags.
SAVE_OUTPUTS_TO_GOOGLE_DRIVE = False  # @param {type:"boolean"}
AUTO_DOWNLOAD_ZIP = True  # @param {type:"boolean"}
SAVE_TXT = True
SAVE_MD = True
SAVE_JSON = True
SAVE_CSV = True
SAVE_XLSX = True
SAVE_SRT = True
INCLUDE_TIMESTAMPS_IN_TXT = True

# Advanced
REQUIRE_CUDA = True
UPLOAD_DIR = "/content/haas_ja_uploads"
WORK_DIR = "/content/haas_ja_work"
OUTPUT_DIR = "/content/haas_ja_outputs"
ZIP_FILE_NAME = "haas_ja_outputs.zip"

SUPPORTED_EXTS = {
    ".aac", ".avi", ".flac", ".m4a", ".m4v", ".mkv", ".mov",
    ".mp3", ".mp4", ".ogg", ".opus", ".wav", ".webm", ".wma"
}

RETURN_TIMESTAMPS = TIMESTAMP_MODE == "subtitle"

if TIMESTAMP_MODE not in {"subtitle", "none"}:
    raise ValueError("TIMESTAMP_MODE must be subtitle / none.")
if not 0 < CHUNK_SECONDS <= 28:
    raise ValueError("CHUNK_SECONDS must be between 0 and 28.")

for p in [UPLOAD_DIR, WORK_DIR, OUTPUT_DIR]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("An loda saituna.")


In [ ]:
# @title 2. Zaɓin fayil / mount na Google Drive
# ============================================================
# 2. Zaɓin fayil / mount na Google Drive
# ============================================================
from pathlib import Path
import shutil

def collect_supported_files(root: Path):
    if root.is_file():
        files = [root]
    elif root.is_dir():
        pattern = "**/*" if RECURSIVE else "*"
        files = [p for p in root.glob(pattern) if p.is_file()]
    else:
        raise FileNotFoundError(f"Input path ba ya wanzu: {root}")
    files = [p for p in files if p.suffix.lower() in SUPPORTED_EXTS]
    if FILENAME_CONTAINS:
        files = [p for p in files if FILENAME_CONTAINS in p.name]
    return sorted(files)

selected_files = []

if INPUT_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    upload_dir = Path(UPLOAD_DIR)
    for name, data in uploaded.items():
        target = upload_dir / Path(name).name
        target.write_bytes(data)
        if target.suffix.lower() in SUPPORTED_EXTS:
            selected_files.append(target)
elif INPUT_MODE == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    root = Path(GOOGLE_DRIVE_ROOT)
    if GOOGLE_DRIVE_SUBPATH:
        root = root / GOOGLE_DRIVE_SUBPATH
    selected_files = collect_supported_files(root)
else:
    raise ValueError("INPUT_MODE dole ne ya zama upload ko google_drive.")

if not selected_files:
    raise RuntimeError("Ba a zaɓi supported audio/video files ba.")

print("Fayilolin da aka zaɓa:")
for p in selected_files:
    print(" -", p)


In [ ]:
# @title 3. Duba GPU da shigar da dependencies
# ============================================================
# 3. Duba GPU da shigar da dependencies
# ============================================================
import os, shutil, subprocess, sys
from pathlib import Path

if os.environ.get("USE_TF") != "0":
    raise RuntimeError("Run section 1 first in a fresh runtime (USE_TF=0 must be set before Transformers imports).")

import torch

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("GPU ba ya samuwa. A Colab, zaɓi “Runtime → Change runtime type → GPU”.")

print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU compute capability:", torch.cuda.get_device_capability(0))
print("Ana goyon bayan BF16:", torch.cuda.is_bf16_supported())

if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.check_call(["apt-get", "-qq", "update"])
    subprocess.check_call(["apt-get", "-qq", "install", "-y", "ffmpeg"])

# Official Haas-ASR source pinned to a reviewed upstream commit.
# Keep the installer and the preflight on the same Transformers version.
TRANSFORMERS_VERSION = "4.57.6"
install_command = [
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--progress-bar", "off",
    "haas-asr @ git+https://github.com/third-intelligence/Haas-ASR.git@acc614f68769dcbfae9dba96b48439c4a66295af",
    f"transformers=={TRANSFORMERS_VERSION}", "pandas", "openpyxl",
]
# Preserve the real pip error rather than a quiet CalledProcessError.
installation = subprocess.run(
    install_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
log_path = Path("/content/haas_ja_pip_install.log")
log_path.write_text(installation.stdout or "", encoding="utf-8")
if installation.returncode != 0:
    print(f"pip install failed (exit {installation.returncode}). Full log: {log_path}")
    print("\n".join((installation.stdout or "").splitlines()[-120:]))
    raise RuntimeError(f"Dependency installation failed; full pip log: {log_path}")
print("An kammala shigar da dependencies.")
print("Full pip log:", log_path)

# Check the decoder in a fresh Python interpreter before downloading the model.
preflight = (
    "import importlib.metadata, os, sys; "
    "assert os.environ.get('USE_TF') == '0', 'USE_TF must be 0'; "
    f"assert importlib.metadata.version('transformers') == {TRANSFORMERS_VERSION!r}; "
    "from transformers.utils import is_tf_available; "
    "assert not is_tf_available(), 'TensorFlow import must be disabled'; "
    "import haas_asr; from haas_asr._fla import check_fla_version; check_fla_version(); "
    "from fla.layers.kda import KimiDeltaAttention; "
    "assert 'tensorflow' not in sys.modules, 'TensorFlow loaded unexpectedly'"
)
subprocess.check_call([sys.executable, "-c", preflight])
print("Decoder import check passed. Continue to model loading.")


In [ ]:
# @title 4. Ayyukan taimako
# ============================================================
# 4. Ayyukan taimako
# ============================================================
from pathlib import Path
import csv, json, math, re, shutil, subprocess
import pandas as pd

ASCII_WORD = re.compile(r"[A-Za-z0-9]")

def safe_stem(path: Path) -> str:
    name = re.sub(r'[\\/:*?"<>|]+', "_", path.stem).strip()
    return name or "transcription"

def run_command(cmd):
    cp = subprocess.run(cmd, capture_output=True, text=True)
    if cp.returncode != 0:
        raise RuntimeError((cp.stderr or cp.stdout or "Aiwatar da command ya gaza.").strip())
    return cp

def convert_to_mono_16k_wav(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    run_command([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(src), "-vn", "-ac", "1", "-ar", "16000",
        "-c:a", "pcm_s16le", str(dst)
    ])
    if not dst.exists() or dst.stat().st_size == 0:
        raise RuntimeError(f"Cire audio ya gaza: {src}")

def join_text_parts(parts):
    result = ""
    for part in parts:
        part = (part or "").strip()
        if not part:
            continue
        if result and ASCII_WORD.search(result[-1]) and ASCII_WORD.search(part[0]):
            result += " "
        result += part
    return result

def estimated_segment(text: str, start: float, end: float, chunk_index: int):
    """Approximate subtitle span: chunk boundaries, not forced alignment."""
    return {
        "type": "segment", "text": text, "start": start, "end": end,
        "duration": end - start, "chunk_index": chunk_index,
        "timing_source": "approximate_chunk_bounds",
    }

def build_subtitle_cues(all_ts):
    if TIMESTAMP_MODE != "subtitle":
        return [], "none"
    cues = [
        {"start": x["start"], "end": x["end"], "text": x["text"]}
        for x in all_ts["segment"] if x["text"].strip()
    ]
    return cues, "approximate_chunk_bounds"

def format_srt_timestamp(seconds):
    ms = max(0, int(round(float(seconds) * 1000)))
    h, rem = divmod(ms, 3600000)
    m, rem = divmod(rem, 60000)
    s, ms = divmod(rem, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def wrap_cue_text(text, max_line_chars):
    text = (text or "").strip()
    if not text or len(text) <= max_line_chars:
        return text
    target = len(text) // 2
    candidates = [i + 1 for i, ch in enumerate(text) if ch in "、。！？!?，,．."]
    candidates = [i for i in candidates if 1 <= i < len(text)]
    split = min(candidates, key=lambda i: abs(i - target)) if candidates else min(max_line_chars, len(text)-1)
    return text[:split].rstrip() + "\n" + text[split:].lstrip()

def cues_to_srt(cues):
    blocks = []
    for i, cue in enumerate(cues, 1):
        blocks.append(
            f"{i}\n"
            f"{format_srt_timestamp(cue['start'])} --> {format_srt_timestamp(cue['end'])}\n"
            f"{wrap_cue_text(cue['text'], SUBTITLE_MAX_LINE_CHARS)}"
        )
    return "\n\n".join(blocks) + ("\n" if blocks else "")

def flatten_timestamps(all_ts):
    rows = []
    for kind in ("segment", "word", "char"):
        rows.extend(all_ts[kind])
    return rows

def write_outputs(source_path, payload, out_dir):
    stem = safe_stem(source_path)
    paths = []
    text = payload["text"]
    cues = payload["subtitle_cues"]

    if SAVE_TXT:
        p = out_dir / f"{stem}.txt"
        parts = [text]
        if INCLUDE_TIMESTAMPS_IN_TXT and cues:
            parts += ["", "=== Timestamps ==="]
            parts += [
                f"[{format_srt_timestamp(c['start'])} --> {format_srt_timestamp(c['end'])}] {c['text']}"
                for c in cues
            ]
        p.write_text("\n".join(parts) + "\n", encoding="utf-8")
        paths.append(p)

    if SAVE_MD:
        p = out_dir / f"{stem}.md"
        body = f"# {source_path.name}\n\n{text}\n"
        if cues:
            body += "\n## Timestamps\n\n"
            body += "\n".join(
                f"- `{format_srt_timestamp(c['start'])} --> {format_srt_timestamp(c['end'])}` {c['text']}"
                for c in cues
            )
            body += "\n"
        p.write_text(body, encoding="utf-8")
        paths.append(p)

    if SAVE_JSON:
        p = out_dir / f"{stem}.json"
        p.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        paths.append(p)

    rows = flatten_timestamps(payload["timestamps"])
    if SAVE_CSV:
        p = out_dir / f"{stem}.csv"
        pd.DataFrame(rows, columns=["type","text","start","end","duration","chunk_index"]).to_csv(
            p, index=False, encoding="utf-8-sig"
        )
        paths.append(p)

    if SAVE_XLSX:
        p = out_dir / f"{stem}.xlsx"
        with pd.ExcelWriter(p, engine="openpyxl") as writer:
            for kind in ("segment", "word", "char"):
                pd.DataFrame(payload["timestamps"][kind]).to_excel(
                    writer, sheet_name=kind + "s", index=False
                )
            pd.DataFrame(payload["chunks"]).to_excel(writer, sheet_name="chunks", index=False)
            pd.DataFrame(cues).to_excel(writer, sheet_name="subtitles", index=False)
        paths.append(p)

    if SAVE_SRT and TIMESTAMP_MODE == "subtitle" and cues:
        p = out_dir / f"{stem}.srt"
        p.write_text(cues_to_srt(cues), encoding="utf-8")
        paths.append(p)

    return paths


In [ ]:
# @title 5. Loda Haas Japanese
# ============================================================
# 5. Loda Haas Japanese
# ============================================================
import torch
import haas_asr

torch.set_float32_matmul_precision("high")
# Use CUDA compute capability: T4 (SM 7.5) has no native BF16 even if a support flag says True.
model_dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print("Ana loda model:", MODEL_ID, "dtype:", model_dtype)
asr_model = haas_asr.load_model(MODEL_ID, device="cuda", dtype=model_dtype)
print("An loda model.")


In [ ]:
# @title 6. Transkripsan
# ============================================================
# 6. Transkripsan
# ============================================================
import json, traceback
from pathlib import Path
import haas_asr

output_root = Path(OUTPUT_DIR)
output_root.mkdir(parents=True, exist_ok=True)
manifest = []

for source_path in selected_files:
    source_path = Path(source_path)
    print("\n=== An fara sarrafawa:", source_path.name, "===")
    item_work = Path(WORK_DIR) / safe_stem(source_path)
    item_work.mkdir(parents=True, exist_ok=True)
    wav_path = item_work / "source_16k_mono.wav"
    record = {"source": source_path.name, "status": "running"}

    try:
        convert_to_mono_16k_wav(source_path, wav_path)
        audio = haas_asr.load_audio(str(wav_path))
        duration = len(audio) / haas_asr.SAMPLE_RATE
        bounds = haas_asr.split_at_quiet_points(
            audio, chunk_seconds=CHUNK_SECONDS,
            search_seconds=2.0,
        )
        if not bounds:
            raise RuntimeError("Haas returned no transcription result.")

        chunk_records = []
        chunk_texts = []
        all_ts = {"char": [], "word": [], "segment": []}
        for idx, (first, last) in enumerate(bounds):
            start = first / haas_asr.SAMPLE_RATE
            end = last / haas_asr.SAMPLE_RATE
            print(f"  chunk {idx+1}/{len(bounds)}: {start:.1f}s - {end:.1f}s")
            # Inference is owned by the official Haas API; no older NeMo
            # TDT/CTC decoder, CUDA-graph patch or timestamp flags survive.
            text = haas_asr.transcribe(
                asr_model, [audio[first:last]],
                batch_size=INFERENCE_BATCH_SIZE,
            )[0]
            chunk_texts.append(text)
            if RETURN_TIMESTAMPS and text.strip():
                all_ts["segment"].append(estimated_segment(text, start, end, idx))
            chunk_records.append({
                "index": idx, "start": start, "end": end,
                "duration": end - start, "text": text,
                "timestamp_counts": {
                    "char": 0, "word": 0,
                    "segment": int(bool(RETURN_TIMESTAMPS and text.strip())),
                },
            })

        full_text = join_text_parts(chunk_texts)
        cues, cue_source = build_subtitle_cues(all_ts)
        payload = {
            "source": source_path.name,
            "model": MODEL_ID,
            "decoder": "TDT",
            "timestamp_mode": TIMESTAMP_MODE,
            "timestamp_precision": "approximate_chunk_bounds" if RETURN_TIMESTAMPS else "none",
            "duration_seconds": duration,
            "chunk_seconds": CHUNK_SECONDS,
            "text": full_text,
            "chunks": chunk_records,
            "timestamps": all_ts,
            "subtitle_source": cue_source,
            "subtitle_cues": cues,
            "diagnostics": {
                "chunk_count": len(bounds),
                "characters": len(full_text),
                "char_timestamps": 0,
                "word_timestamps": 0,
                "segment_timestamps": len(all_ts["segment"]),
                "subtitle_cues": len(cues),
                "subtitle_max_duration": max(
                    (c["end"] - c["start"] for c in cues), default=0.0
                ),
                "subtitle_max_chars": max((len(c["text"]) for c in cues), default=0),
                "timestamps_monotonic": all(
                    rows[i]["start"] >= rows[i - 1]["start"]
                    for rows in all_ts.values() for i in range(1, len(rows))
                ),
            },
        }
        paths = write_outputs(source_path, payload, output_root)
        print("  Haruffa:", len(full_text))
        print("  native segment timestamps:", 0)
        print("  word timestamps:", 0)
        print("  approximate subtitle cues:", len(cues))
        record.update({
            "status": "ok", "duration_seconds": duration,
            "characters": len(full_text), "chunks": len(bounds),
            "char_timestamps": 0, "word_timestamps": 0,
            "segment_timestamps": len(all_ts["segment"]),
            "subtitle_cues": len(cues), "subtitle_source": cue_source,
            "outputs": [p.name for p in paths],
        })
    except Exception as exc:
        record.update({
            "status": "error", "error_type": type(exc).__name__,
            "error": str(exc), "traceback": traceback.format_exc(),
        })
        print(record["traceback"])
    manifest.append(record)

manifest_path = output_root / "manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("\nAn kammala sarrafawa.")
print("Manifest:", manifest_path)


In [ ]:
# @title 7. Ajiye / sauke ZIP
# ============================================================
# 7. Ajiye / sauke ZIP
# ============================================================
from pathlib import Path
import shutil

output_root = Path(OUTPUT_DIR)
zip_base = Path("/content") / Path(ZIP_FILE_NAME).stem
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=output_root))
print("ZIP:", zip_path)

if SAVE_OUTPUTS_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_out = Path(GOOGLE_DRIVE_ROOT) / "Haas_Japanese_ASR_outputs"
    drive_out.mkdir(parents=True, exist_ok=True)
    target = drive_out / zip_path.name
    shutil.copy2(zip_path, target)
    print("An ajiye zuwa Google Drive:", target)

if AUTO_DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(zip_path))
